# The Transformer — *Attention Is All You Need*

> Tutorial pair for [`transformer.py`](transformer.py). Read
> [`attention.ipynb`](../attention/attention.ipynb) first.

## 1. Intuition
Take multi-head attention, wrap it with a position-wise MLP, a residual
connection and LayerNorm, and stack the block. Add positional encodings (since
attention is order-agnostic) and you get a fully-parallel sequence model that
trains far better than RNNs — the backbone of BERT, GPT, T5, and ViT.

## 2. Concept (the slide)
- **Encoder block:** self-attention → FFN, each with **residual + LayerNorm**.
- **Decoder block:** *masked* self-attention → **cross-attention** to the encoder
  → FFN.
- **Positional encoding** injects order; **causal mask** enforces autoregression.
- **Teacher forcing** during training; **greedy/beam** decoding at inference.
- **noam LR schedule:** warm up, then decay — critical for stable training.

## 3. Math derivation — the pieces

**Positional encoding.** Attention is permutation-equivariant, so we add position
info:
$$PE_{(pos,2i)}=\sin\!\big(pos/10000^{2i/d}\big),\quad
  PE_{(pos,2i+1)}=\cos\!\big(pos/10000^{2i/d}\big).$$
Because $\sin(a{+}b),\cos(a{+}b)$ are linear in $\sin a,\cos a$, a fixed relative
shift $k$ acts as a **linear map** on $PE$ — attention can learn relative offsets.

**Residual + LayerNorm (pre-norm).** Each sublayer computes
$x \leftarrow x + \text{Sublayer}(\text{LN}(x))$. The residual gives the gradient a
direct $+1$ path (the vanishing-gradient fix from ResNet), and
$\text{LN}(x)=\gamma\frac{x-\mu}{\sqrt{\sigma^2+\epsilon}}+\beta$ (statistics over
the feature dim, **per token**) keeps activations well-scaled regardless of batch.

**Position-wise FFN.** $\text{FFN}(x)=\max(0,xW_1+b_1)W_2+b_2$ applied identically
at every position — the per-token "compute" between attention mixing steps.

**Masked self-attention.** In the decoder, a lower-triangular mask blocks future
positions so the model is a valid autoregressive factorization
$p(y)=\prod_t p(y_t\mid y_{<t},x)$. Training uses **teacher forcing**: feed the
true $y_{<t}$ in parallel and predict all $y_t$ at once.

**noam learning-rate schedule.**
$$\text{lr}(t)=d_{\text{model}}^{-1/2}\cdot\min\!\big(t^{-1/2},\, t\cdot t_{\text{warmup}}^{-3/2}\big).$$
It rises linearly for `warmup` steps then decays as $t^{-1/2}$. Warmup prevents
huge early updates from destabilizing the freshly-initialized attention; the decay
anneals to a good minimum. (See `06.training-techniques/README.md`.)

## 4. NumPy — positional encoding (the closed-form bit)

In [ ]:
# ===== actual implementation from transformer.py =====
from __future__ import annotations

import math

import numpy as np

SEED = 0

def positional_encoding(L, d_model):
    r"""
    PE[pos, 2i]   = sin(pos / 10000^{2i/d})
    PE[pos, 2i+1] = cos(pos / 10000^{2i/d})
    Each dimension is a sinusoid of a different wavelength; relative positions are
    then linear functions of these, so attention can learn to shift by an offset.
    """
    pos = np.arange(L)[:, None]
    i = np.arange(d_model)[None, :]
    angle = pos / np.power(10000, (2 * (i // 2)) / d_model)
    pe = np.zeros((L, d_model))
    pe[:, 0::2] = np.sin(angle[:, 0::2])
    pe[:, 1::2] = np.cos(angle[:, 1::2])
    return pe

## 5. PyTorch — attention, encoder/decoder blocks, full model

In [ ]:
# ===== actual implementation from transformer.py =====
import torch

import torch.nn as nn

import torch.nn.functional as F

def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512):
        super().__init__()
        self.register_buffer("pe", torch.tensor(
            positional_encoding(max_len, d_model), dtype=torch.float32))

    def forward(self, x):                       # x: (B, L, d)
        return x + self.pe[:x.size(1)]

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        self.h, self.d_k = n_heads, d_model // n_heads
        self.qkv = nn.ModuleList(nn.Linear(d_model, d_model) for _ in range(3))
        self.out = nn.Linear(d_model, d_model)
        self.drop = nn.Dropout(dropout)

    def forward(self, q, k, v, mask=None):
        B = q.size(0)
        def split(x, lin): return lin(x).view(B, -1, self.h, self.d_k).transpose(1, 2)
        Q, K, V = (split(t, lin) for t, lin in zip((q, k, v), self.qkv))
        scores = Q @ K.transpose(-1, -2) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        attn = self.drop(scores.softmax(-1))
        out = (attn @ V).transpose(1, 2).contiguous().view(B, -1, self.h * self.d_k)
        return self.out(out)

class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_model, d_ff), nn.ReLU(),
                                 nn.Dropout(dropout), nn.Linear(d_ff, d_model))

    def forward(self, x): return self.net(x)

class EncoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.ff = FeedForward(d_model, d_ff, dropout)
        self.n1, self.n2 = nn.LayerNorm(d_model), nn.LayerNorm(d_model)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, mask=None):            # pre-norm + residual
        x = x + self.drop(self.attn(self.n1(x), self.n1(x), self.n1(x), mask))
        x = x + self.drop(self.ff(self.n2(x)))
        return x

class DecoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.cross_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.ff = FeedForward(d_model, d_ff, dropout)
        self.n1 = nn.LayerNorm(d_model); self.n2 = nn.LayerNorm(d_model)
        self.n3 = nn.LayerNorm(d_model); self.drop = nn.Dropout(dropout)

    def forward(self, x, mem, tgt_mask=None, src_mask=None):
        x = x + self.drop(self.self_attn(self.n1(x), self.n1(x), self.n1(x), tgt_mask))
        nx = self.n2(x)
        x = x + self.drop(self.cross_attn(nx, mem, mem, src_mask))   # attend to encoder
        x = x + self.drop(self.ff(self.n3(x)))
        return x

class Transformer(nn.Module):
    def __init__(self, src_vocab, tgt_vocab, d_model=64, n_heads=4,
                 d_ff=128, n_layers=2, dropout=0.1, max_len=64):
        super().__init__()
        self.src_emb = nn.Embedding(src_vocab, d_model)
        self.tgt_emb = nn.Embedding(tgt_vocab, d_model)
        self.pos = PositionalEncoding(d_model, max_len)
        self.enc = nn.ModuleList(EncoderLayer(d_model, n_heads, d_ff, dropout)
                                 for _ in range(n_layers))
        self.dec = nn.ModuleList(DecoderLayer(d_model, n_heads, d_ff, dropout)
                                 for _ in range(n_layers))
        self.fc = nn.Linear(d_model, tgt_vocab)
        self.d_model = d_model

    def encode(self, src, src_mask=None):
        x = self.pos(self.src_emb(src) * math.sqrt(self.d_model))
        for layer in self.enc:
            x = layer(x, src_mask)
        return x

    def decode(self, tgt, mem, tgt_mask=None, src_mask=None):
        x = self.pos(self.tgt_emb(tgt) * math.sqrt(self.d_model))
        for layer in self.dec:
            x = layer(x, mem, tgt_mask, src_mask)
        return self.fc(x)

    def forward(self, src, tgt, src_mask=None, tgt_mask=None):
        return self.decode(tgt, self.encode(src, src_mask), tgt_mask, src_mask)

def causal_mask(L, device):
    return torch.tril(torch.ones(L, L, device=device)).bool()

def noam_lr(step, d_model, warmup):
    """LR ∝ d_model^-0.5 * min(step^-0.5, step*warmup^-1.5). Warmup then decay."""
    step = max(step, 1)
    return d_model ** -0.5 * min(step ** -0.5, step * warmup ** -1.5)

def make_reverse_data(n, L, vocab, seed=SEED):
    """src = random tokens; tgt = src reversed. Tokens 0=PAD,1=BOS,2=EOS."""
    rng = np.random.default_rng(seed)
    src = rng.integers(3, vocab, size=(n, L))
    tgt_out = src[:, ::-1]
    bos = np.full((n, 1), 1); eos = np.full((n, 1), 2)
    tgt_in = np.concatenate([bos, tgt_out], 1)          # teacher forcing input
    tgt_out = np.concatenate([tgt_out, eos], 1)         # shifted target
    return src, tgt_in, tgt_out

def demo():
    torch.manual_seed(SEED); np.random.seed(SEED)
    # On many-core CPUs, default multi-threading thrashes on these tiny ops;
    # pinning to 1 thread makes the demo ~10x faster and avoids timeouts.
    torch.set_num_threads(1)
    dev = get_device()
    V, L = 14, 6
    src, tin, tout = make_reverse_data(256, L, V)   # tiny: trivial task, CPU-fast
    src = torch.tensor(src, device=dev)
    tin = torch.tensor(tin, device=dev)
    tout = torch.tensor(tout, device=dev)

    model = Transformer(V, V, d_model=64, n_heads=4, n_layers=2).to(dev)
    opt = torch.optim.Adam(model.parameters(), lr=1.0, betas=(0.9, 0.98), eps=1e-9)
    loss_fn = nn.CrossEntropyLoss(ignore_index=0)

    model.train()
    for step in range(1, 501):
        for g in opt.param_groups:               # noam schedule (warmup + decay)
            g["lr"] = noam_lr(step, 64, warmup=200)
        tmask = causal_mask(tin.size(1), dev)
        logits = model(src, tin, tgt_mask=tmask)
        loss = loss_fn(logits.reshape(-1, V), tout.reshape(-1))
        opt.zero_grad(); loss.backward(); opt.step()
        if step % 150 == 0:
            print(f"step {step:4d}  loss {loss.item():.3f}  lr {opt.param_groups[0]['lr']:.4f}")

    # greedy decode one example
    model.eval()
    with torch.no_grad():
        s = src[:1]
        mem = model.encode(s)
        ys = torch.tensor([[1]], device=dev)                     # BOS
        for _ in range(L):
            m = causal_mask(ys.size(1), dev)
            logit = model.decode(ys, mem, tgt_mask=m)
            nxt = logit[:, -1].argmax(-1, keepdim=True)
            ys = torch.cat([ys, nxt], 1)
    print("\nsrc      :", s[0].tolist())
    print("reversed :", s[0].flip(0).tolist())
    print("predicted:", ys[0, 1:].tolist())

## 6. Train a seq2seq **reverse** task (teacher forcing + noam) and decode

In [ ]:
# ~30–60s on CPU: trains 1000 steps then greedily decodes one example.
import transformer as M
M.demo()

## 7. Visualization — positional encoding & the noam schedule

In [ ]:
import numpy as np, matplotlib.pyplot as plt
import transformer as M

pe = M.positional_encoding(80, 64)
steps = np.arange(1, 4000)
lr = [M.noam_lr(s, 64, warmup=300) for s in steps]

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
im = ax[0].imshow(pe.T, aspect="auto", cmap="RdBu")
ax[0].set_xlabel("position"); ax[0].set_ylabel("dimension")
ax[0].set_title("Sinusoidal positional encoding"); fig.colorbar(im, ax=ax[0])
ax[1].plot(steps, lr); ax[1].axvline(300, ls="--", c="r", label="warmup end")
ax[1].set_xlabel("step"); ax[1].set_ylabel("learning rate")
ax[1].set_title("noam schedule (warmup → decay)"); ax[1].legend()
plt.tight_layout(); plt.show()

## 8. Takeaways & what's next
- The Transformer = attention + FFN + **residual + LayerNorm**, stacked, with
  positional encodings and careful **LR warmup**.
- Masking turns the decoder into an autoregressive model; teacher forcing makes
  training parallel.
- Specializations (next files in `architectures/`, see `MAP.md`):
  - **BERT** = encoder-only, masked-LM pretraining (bidirectional).
  - **GPT** = decoder-only, causal LM (generation).
  - **ViT** = patches-as-tokens for images.